In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = "sk-proj-ORV1HwUGs8R8vWlaRYLdAzJfhxLH9NYWyb5GDGGm9Il4JLsPWQX5L1I8A9hR_Cbs1a0JWaPCtUdqvv5LRedP2mIWl8A"

## Install libraries

In [ ]:
# !pip install -q youtube-transcript-api langchain-community langchain-openai \
#                faiss-cpu tiktoken python-dotenv

In [23]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

## Step 1a - Indexing (Document Ingestion)

In [7]:

# modern way use the oop approach
video_id = "Gfr50f6ZBvo"

try:
    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(video_id, languages=["en"])

    transcript = " ".join(snippet.text for snippet in fetched_transcript)

    # print(transcript)

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")

except NoTranscriptFound:
    print("No English transcript found.")

In [ ]:
fetched_transcript

## Step 1b - Indexing (Text Splitting)

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [13]:
len(chunks)

168

In [14]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

In [15]:
from dotenv import load_dotenv

load_dotenv()

True

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [25]:
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1985.71it/s]


In [28]:
vector_store.index_to_docstore_id

{0: '310b9775-bd21-47b5-a612-c8b207041af2',
 1: '1ca068e2-aa6e-4773-bc28-cfa674193380',
 2: '26bd410c-166a-4686-83bb-ec2b28308467',
 3: '721b7e34-5b7d-44ce-9c45-c75390088708',
 4: '2cae260d-cd9c-47d9-b640-91512e05dc0e',
 5: '0e31dede-e8ec-4b91-908d-217f7b51f0fc',
 6: '01dc8a7e-222f-4e54-ab69-e536560fe143',
 7: 'f5517984-7298-4f24-9785-b0cc0fb8be70',
 8: 'ae8c73c7-5d7e-41b1-bd54-f3f30de14d68',
 9: '2b6adf95-12b1-4374-8f4b-133da0332b15',
 10: '943ce12e-9edf-4b87-af24-d590b582fecc',
 11: '7524dc72-784e-418d-9a1e-437ef8662238',
 12: '3a44b177-eb67-4b42-b083-57d39feea73e',
 13: 'f5884bba-9c9c-46d6-b924-5f34bc35fc75',
 14: '4371fba5-c25e-437c-b191-2d6132b1a818',
 15: 'c5ac45d6-752e-4307-b47d-089573f31a41',
 16: '7fdb17b6-8490-4203-b25f-58bc35ebf98a',
 17: '04ec1c4f-afea-48d0-bc28-c5bfa8fe9dc9',
 18: 'e94f3610-8dec-4204-a674-d160ed84e16a',
 19: '7ac24035-6f82-4b96-890b-965d28c0f0df',
 20: '67dd2079-6be9-4a7d-98c5-683ce8c9b965',
 21: 'aa85880f-86d5-4d03-83c2-896c9b943532',
 22: '8bc984d0-74f5-

In [ ]:
vector_store.get_by_ids(['80fa4ac0-0f85-438a-ba37-819499a385e6'])

[Document(id='80fa4ac0-0f85-438a-ba37-819499a385e6', metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time')]

## Step 2 - Retrieval

In [31]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [32]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f5d18aa2cf0>, search_kwargs={'k': 4})

In [33]:
retriever.invoke('What is deepmind')

[Document(id='fbbed96a-9978-4b76-849a-dc2860b02385', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')